In [13]:
import json

notebook_path = '/content/Bài 3.Xây dựng mô hình phân loại cho dữ liệu bảng.ipynb.ipynb'

try:
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb_data = json.load(f)

    print(f"Notebook version: {nb_data.get('nbformat')}.{nb_data.get('nbformat_minor')}")
    cells = nb_data.get('cells', [])
    print(f"Total cells: {len(cells)}")

    # Display first few cell types and a preview of their content to locate exercises
    for idx, cell in enumerate(cells[:15]):
        cell_type = cell.get('cell_type')
        source = ''.join(cell.get('source', []))
        preview = source[:150].replace('\n', ' ')
        print(f"[{idx}] Type: {cell_type} | Preview: {preview}...")
except Exception as e:
    print(f"Error reading notebook: {e}")

Notebook version: 4.5
Total cells: 153
[0] Type: markdown | Preview: # THỰC HÀNH XÂY DỰNG MÔ HÌNH CLASSIFICATION VỚI DỮ LIỆU BẢNG  ### Bài toán Xây dựng mô hình **phân loại nhị phân** để nhận diện mẫu thuộc lớp **malign...
[1] Type: markdown | Preview: # PHẦN 1. CHUẨN BỊ MÔI TRƯỜNG...
[2] Type: markdown | Preview: ## Bước 1. Cài đặt và import thư viện  Các thư viện chính:  - `numpy`, `pandas`: xử lý dữ liệu. - `matplotlib`: trực quan hóa. - `scikit-learn`: datas...
[3] Type: code | Preview: import warnings warnings.filterwarnings("ignore")  import numpy as np import pandas as pd import matplotlib.pyplot as plt  from sklearn.datasets impor...
[4] Type: markdown | Preview: ## Bước 2. Nạp dữ liệu Breast Cancer Wisconsin  Bộ dữ liệu gồm:  - **569 mẫu** trong dữ liệu gốc. - **30 đặc trưng số**. - Hai lớp:   - `malignant`: á...
[5] Type: code | Preview: cancer = load_breast_cancer(as_frame=True)  X_full = cancer.data.copy()  y_full = (cancer.target == 0).astype(int)  df_full = X_full.copy() 

In [14]:
import json

notebook_path = '/content/Bài 3.Xây dựng mô hình phân loại cho dữ liệu bảng.ipynb.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

cells = nb_data.get('cells', [])

# Search for exercises, TODOs, and assignments
keywords = ['todo', 'yêu cầu', 'bài tập', 'thực hành', 'exercise', 'nhiệm vụ', 'chưa hoàn thành']
for idx, cell in enumerate(cells):
    source = ''.join(cell.get('source', []))
    lower_source = source.lower()

    # If the cell contains keywords or is a code cell that is empty or contains placeholders
    if any(kw in lower_source for kw in keywords):
        print(f"===[ Cell #{idx} ({cell.get('cell_type')}) ]===")
        print(source.strip())
        print("-" * 50)

===[ Cell #0 (markdown) ]===
# THỰC HÀNH XÂY DỰNG MÔ HÌNH CLASSIFICATION VỚI DỮ LIỆU BẢNG

### Bài toán
Xây dựng mô hình **phân loại nhị phân** để nhận diện mẫu thuộc lớp **malignant** từ bộ dữ liệu Breast Cancer Wisconsin của `scikit-learn`.

Bài này đi vào quy trình xây dựng và đánh giá mô hình classification khi dữ liệu bị **class imbalance**.

### Mục tiêu h
- Nhận biết một bài toán **binary classification**.
- Phân tích mức độ **mất cân bằng lớp**.
- Hiểu vì sao **Accuracy có thể gây hiểu nhầm** khi dữ liệu imbalance.
- Chia dữ liệu bằng `train_test_split(..., stratify=y)`.
- Hiểu và sử dụng **StandardScaler** đúng cách.
- Hiểu nguyên lý **K-Fold Cross Validation**.
- Phân biệt `KFold` và `StratifiedKFold`.
- Đánh giá mô hình bằng Accuracy, Precision, Recall, F1-score, ROC-AUC và PR-AUC.
- Áp dụng:
  - Random Over-Sampling;
  - Random Under-Sampling;
  - SMOTE.
- Kết hợp chuẩn hóa + sampling + model bằng **Pipeline** để tránh **data leakage**.
- So sánh các phương án bằng Cross Va

In [15]:
import json

notebook_path = '/content/Bài 3.Xây dựng mô hình phân loại cho dữ liệu bảng.ipynb.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

cells = nb_data.get('cells', [])

print(f"Retrieving cells from index 91 to {len(cells)-1}:")
for idx in range(91, len(cells)):
    cell = cells[idx]
    print(f"\n=== Cell #{idx} ({cell.get('cell_type')}) ===")
    print(''.join(cell.get('source', [])))
    print("="*40)

Retrieving cells from index 91 to 152:

=== Cell #91 (markdown) ===
# PHẦN 11. BÀI TẬP THỰC HÀNH

=== Cell #92 (markdown) ===
### Bài 1 — Mức độ imbalance
Thay số mẫu malignant từ `60` thành:

- `40`;
- `120`;
- toàn bộ dữ liệu gốc.

So sánh Accuracy, Recall, F1 và PR-AUC.

---

### Bài 2 — KFold và StratifiedKFold
Dùng cùng Logistic Regression và so sánh:

- mean score;
- standard deviation giữa các fold.

Giải thích tại sao `StratifiedKFold` thường ổn định hơn với dữ liệu imbalance.

---

### Bài 3 — Thay đổi số fold
Lần lượt dùng:

- `K = 3`;
- `K = 5`;
- `K = 10`.

Quan sát sự thay đổi của mean và standard deviation.

---

### Bài 4 — Thay đổi mức Over-Sampling
Thử:

```python
RandomOverSampler(sampling_strategy=0.5)
RandomOverSampler(sampling_strategy=0.75)
RandomOverSampler(sampling_strategy=1.0)
```

Giải thích trade-off giữa Precision và Recall.

---

### Bài 5 — So sánh ba kỹ thuật sampling
So sánh:

- Random Over-Sampling;
- Random Under-Sampling;
- SMOTE.

Không chỉ chọn kỹ 

In [16]:
import json

notebook_path = '/content/Bài 3.Xây dựng mô hình phân loại cho dữ liệu bảng.ipynb.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

# Prepare new cells to be appended
new_cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# PHẦN 12. LỜI GIẢI CHI TIẾT CÁC BÀI TẬP THỰC HÀNH\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 1 — Mức độ imbalance\n",
            "Thay số mẫu malignant từ 60 thành 40, 120, và toàn bộ dữ liệu gốc. So sánh Accuracy, Recall, F1 và PR-AUC."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import numpy as np\n",
            "import pandas as pd\n",
            "from sklearn.datasets import load_breast_cancer\n",
            "from sklearn.model_selection import train_test_split\n",
            "from sklearn.preprocessing import StandardScaler\n",
            "from sklearn.linear_model import LogisticRegression\n",
            "from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_recall_curve, auc\n",
            "\n",
            "RANDOM_STATE = 42\n",
            "cancer = load_breast_cancer(as_frame=True)\n",
            "X_full = cancer.data.copy()\n",
            "y_full = (cancer.target == 0).astype(int) # 1 if malignant (cancer.target == 0 in original sklearn representation is malignant)\n",
            "df_full = X_full.copy()\n",
            "df_full['target'] = y_full\n",
            "\n",
            "benign_df = df_full[df_full['target'] == 0]\n",
            "malignant_full = df_full[df_full['target'] == 1]\n",
            "\n",
            "scenarios = {\n",
            "    '40 malignant': 40,\n",
            "    '120 malignant': 120,\n",
            "    'Full dataset': len(malignant_full)\n",
            "}\n",
            "\n",
            "results = []\n",
            "for label, n_mal in scenarios.items():\n",
            "    if label == 'Full dataset':\n",
            "        df_scenario = df_full.copy()\n",
            "    else:\n",
            "        malignant_sampled = malignant_full.sample(n=n_mal, random_state=RANDOM_STATE)\n",
            "        df_scenario = pd.concat([benign_df, malignant_sampled], axis=0)\n",
            "    \n",
            "    X = df_scenario.drop(columns=['target'])\n",
            "    y = df_scenario['target']\n",
            "    \n",
            "    X_train, X_test, y_train, y_test = train_test_split(\n",
            "        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y\n",
            "    )\n",
            "    \n",
            "    scaler = StandardScaler()\n",
            "    X_train_scaled = scaler.fit_transform(X_train)\n",
            "    X_test_scaled = scaler.transform(X_test)\n",
            "    \n",
            "    model = LogisticRegression(random_state=RANDOM_STATE)\n",
            "    model.fit(X_train_scaled, y_train)\n",
            "    y_pred = model.predict(X_test_scaled)\n",
            "    y_proba = model.predict_proba(X_test_scaled)[:, 1]\n",
            "    \n",
            "    precision, recall, _ = precision_recall_curve(y_test, y_proba)\n            ",
            "    pr_auc = auc(recall, precision)\n",
            "    \n",
            "    results.append({\n",
            "        'Scenario': label,\n",
            "        'Accuracy': accuracy_score(y_test, y_pred),\n",
            "        'Recall': recall_score(y_test, y_pred),\n",
            "        'F1-Score': f1_score(y_test, y_pred),\n",
            "        'PR-AUC': pr_auc\n",
            "    })\n",
            "\n",
            "df_results = pd.DataFrame(results)\n",
            "print(df_results.to_markdown(index=False))\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Nhận xét Bài 1:\n",
            "- Khi số mẫu malignant (lớp thiểu số) tăng lên, sự mất cân bằng giảm đi.\n",
            "- **Recall** và **F1-Score** có xu hướng tăng và ổn định hơn vì mô hình học được nhiều đặc trưng hơn của lớp thiểu số.\n",
            "- **PR-AUC** cũng tăng rõ rệt khi lượng mẫu malignant tăng, phản ánh khả năng phân biệt lớp tích cực chính xác hơn."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 2 — KFold và StratifiedKFold\n",
            "Dùng cùng Logistic Regression và so sánh mean score và standard deviation giữa các fold. Giải thích tại sao StratifiedKFold ổn định hơn."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score\n",
            "\n",
            "# Sử dụng kịch bản mất cân bằng với 60 mẫu malignant như trong bài học\n",
            "malignant_sampled = malignant_full.sample(n=60, random_state=RANDOM_STATE)\n",
            "df_imbalanced = pd.concat([benign_df, malignant_sampled], axis=0).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)\n",
            "X_imb = df_imbalanced.drop(columns=['target'])\n",
            "y_imb = df_imbalanced['target']\n",
            "\n",
            "scaler = StandardScaler()\n",
            "X_imb_scaled = scaler.fit_transform(X_imb)\n",
            "\n",
            "kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
            "skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
            "\n",
            "model = LogisticRegression(random_state=RANDOM_STATE)\n",
            "\n",
            "kf_scores = cross_val_score(model, X_imb_scaled, y_imb, cv=kf, scoring='f1')\n",
            "skf_scores = cross_val_score(model, X_imb_scaled, y_imb, cv=skf, scoring='f1')\n",
            "\n",
            "print(f\"KFold F1 scores: {kf_scores} | Mean: {kf_scores.mean():.4f} | Std: {kf_scores.std():.4f}\")\n",
            "print(f\"StratifiedKFold F1 scores: {skf_scores} | Mean: {skf_scores.mean():.4f} | Std: {skf_scores.std():.4f}\")\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Giải thích Bài 2:\n",
            "- **StratifiedKFold** giữ nguyên tỷ lệ nhãn lớp (target) trên từng fold giống như phân bố trên toàn bộ dataset gốc.\n",
            "- Trong khi đó, **KFold** chia ngẫu nhiên nên có những fold sẽ nhận rất ít hoặc hầu như không có mẫu của lớp thiểu số (malignant), dẫn đến điểm số F1 dao động cực kỳ mạnh giữa các fold (Std cao). Vì vậy, StratifiedKFold ổn định hơn hẳn (Std thấp hơn)."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 3 — Thay đổi số fold\n",
            "Lần lượt dùng K = 3, K = 5, K = 10. Quan sát sự thay đổi của mean và standard deviation."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "for k in [3, 5, 10]:\n",
            "    skf_k = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)\n",
            "    scores = cross_val_score(model, X_imb_scaled, y_imb, cv=skf_k, scoring='f1')\n",
            "    print(f\"K = {k} | F1 Mean: {scores.mean():.4f} | F1 Std: {scores.std():.4f}\")\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Nhận xét Bài 3:\n",
            "- Khi K nhỏ (K = 3), lượng dữ liệu validation của mỗi fold lớn nhưng số lượng fold đánh giá ít, dẫn đến độ lệch chuẩn có thể nhỏ hoặc lớn tùy phân phối.\n",
            "- Khi K lớn (K = 10), mỗi fold validation có rất ít mẫu. Do đó kết quả từng fold dễ bị ảnh hưởng bởi nhiễu, làm cho Std có xu hướng tăng lên nhẹ nhưng ước lượng mean thường sát thực tế hơn."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 4 — Thay đổi mức Over-Sampling\n",
            "Thử các mức sampling_strategy khác nhau và giải thích trade-off giữa Precision và Recall."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "from imblearn.over_sampling import RandomOverSampler\n",
            "from imblearn.pipeline import Pipeline\n",
            "from sklearn.metrics import precision_score\n",
            "\n",
            "X_train, X_test, y_train, y_test = train_test_split(\n",
            "    X_imb, y_imb, test_size=0.2, random_state=RANDOM_STATE, stratify=y_imb\n",
            ")\n",
            "\n",
            "strategies = [0.5, 0.75, 1.0]\n",
            "for strat in strategies:\n",
            "    pipeline = Pipeline([\n",
            "        ('scaler', StandardScaler()),\n",
            "        ('sampler', RandomOverSampler(sampling_strategy=strat, random_state=RANDOM_STATE)),\n",
            "        ('classifier', LogisticRegression(random_state=RANDOM_STATE))\n",
            "    ])\n",
            "    \n",
            "    pipeline.fit(X_train, y_train)\n",
            "    y_pred = pipeline.predict(X_test)\n",
            "    \n",
            "    prec = precision_score(y_test, y_pred)\n",
            "    rec = recall_score(y_test, y_pred)\n",
            "    print(f\"Strategy: {strat:.2f} | Precision: {prec:.4f} | Recall: {rec:.4f}\")\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Giải thích Bài 4 (Precision-Recall Trade-off):\n",
            "- Khi ta tăng tỷ lệ `sampling_strategy` (từ 0.5 lên 1.0), mô hình sẽ được huấn luyện trên nhiều mẫu tự nhân bản của lớp thiểu số hơn.\n",
            "- Điều này thúc đẩy mô hình ưu tiên dự đoán lớp thiểu số hơn, dẫn đến tăng **Recall** (phát hiện được nhiều ca bệnh thực sự hơn).\n",
            "- Tuy nhiên, việc dự đoán rộng rãi này cũng làm tăng số lượng dự đoán sai (False Positives), dẫn tới **Precision** bị giảm xuống."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 5 — So sánh ba kỹ thuật sampling\n",
            "So sánh: Random Over-Sampling, Random Under-Sampling, SMOTE."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "from imblearn.under_sampling import RandomUnderSampler\n",
            "from imblearn.over_sampling import SMOTE\n",
            "\n",
            "techniques = {\n",
            "    'Random Over-Sampling': RandomOverSampler(random_state=RANDOM_STATE),\n",
            "    'Random Under-Sampling': RandomUnderSampler(random_state=RANDOM_STATE),\n",
            "    'SMOTE': SMOTE(random_state=RANDOM_STATE)\n",
            "}\n",
            "\n",
            "for name, sampler in techniques.items():\n",
            "    pipeline = Pipeline([\n",
            "        ('scaler', StandardScaler()),\n",
            "        ('sampler', sampler),\n",
            "        ('classifier', LogisticRegression(random_state=RANDOM_STATE))\n",
            "    ])\n",
            "    pipeline.fit(X_train, y_train)\n",
            "    y_pred = pipeline.predict(X_test)\n",
            "    y_proba = pipeline.predict_proba(X_test)[:, 1]\n",
            "    \n",
            "    precision, recall, _ = precision_recall_curve(y_test, y_proba)\n",
            "    pr_auc = auc(recall, precision)\n",
            "    \n",
            "    print(f\"[{name}]\")\n",
            "    print(f\"  Accuracy : {accuracy_score(y_test, y_pred):.4f}\")\n",
            "    print(f\"  Precision: {precision_score(y_test, y_pred):.4f}\")\n",
            "    print(f\"  Recall   : {recall_score(y_test, y_pred):.4f}\")\n",
            "    print(f\"  F1-Score : {f1_score(y_test, y_pred):.4f}\")\n",
            "    print(f\"  PR-AUC   : {pr_auc:.4f}\")\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Phân tích Bài 5:\n",
            "- **Random Over-Sampling** nhân bản các mẫu có sẵn nên dễ gây Overfitting.\n",
            "- **Random Under-Sampling** giảm bớt các mẫu đa số nên dễ làm mất thông tin quan trọng của tập dữ liệu.\n",
            "- **SMOTE** sinh các mẫu ảo mới nằm giữa các mẫu láng giềng gần nhất, giúp mô hình học được ranh giới quyết định (decision boundary) tốt hơn, thường đem lại sự cân bằng tốt nhất giữa Precision, Recall và PR-AUC cao nhất."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 6 — Thay classifier\n",
            "Giữ nguyên quy trình, thử KNN, SVM, Decision Tree. Phân tích mô hình nào nhạy với StandardScaler hơn."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "from sklearn.neighbors import KNeighborsClassifier\n",
            "from sklearn.svm import SVC\n",
            "from sklearn.tree import DecisionTreeClassifier\n",
            "\n",
            "classifiers = {\n",
            "    'KNN': KNeighborsClassifier(),\n",
            "    'SVM': SVC(probability=True, random_state=RANDOM_STATE),\n",
            "    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE)\n",
            "}\n",
            "\n",
            "for clf_name, clf in classifiers.items():\n",
            "    for scaled in [True, False]:\n",
            "        steps = []\n",
            "        if scaled:\n",
            "            steps.append(('scaler', StandardScaler()))\n",
            "        steps.append(('classifier', clf))\n",
            "        \n",
            "        pipeline = Pipeline(steps)\n",
            "        pipeline.fit(X_train, y_train)\n",
            "        y_pred = pipeline.predict(X_test)\n",
            "        f1 = f1_score(y_test, y_pred)\n",
            "        \n",
            "        print(f\"Classifier: {clf_name:<15} | Scaled: {str(scaled):<5} | F1-Score: {f1:.4f}\")\n"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "### Phân tích Bài 6:\n",
            "- **KNN** cực kỳ nhạy cảm với tỉ lệ scale vì thuật toán này tính toán khoảng cách Euclidean giữa các điểm dữ liệu. Nếu không chuẩn hóa, các đặc trưng có biên độ lớn sẽ hoàn toàn lấn át.\n",
            "- **SVM** cũng rất nhạy cảm với scale vì SVM tìm kiếm biên phân cách tối đa dựa trên khoảng cách hình học.\n",
            "- **Decision Tree** hoàn toàn không nhạy cảm với StandardScaler vì quyết định rẽ nhánh dựa trên việc tìm ngưỡng phân tách đơn lẻ của từng đặc trưng độc lập, do đó thứ tự trị số không thay đổi khi scale tuyến tính."
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "## Bài 7 — Câu hỏi suy luận (Trả lời ngắn)\n",
            "\n",
            "1. **Tại sao không nên sampling tập test?**\n",
            "   - Tập test cần phản ánh chính xác phân phối thực tế ngoài đời thực. Nếu ta can thiệp sampling tập test, kết quả đánh giá sẽ bị ảo và không thực tế.\n",
            "2. **Tại sao không nên fit StandardScaler trên toàn bộ dữ liệu?**\n",
            "   - Sẽ gây ra hiện tượng **Data Leakage** (rò rỉ dữ liệu từ tập test vào tập train), làm cho mô hình có vẻ tốt lúc học nhưng thực tế kém đi khi test.\n",
            "3. **Khi nào Recall quan trọng hơn Precision?**\n",
            "   - Trong y tế (phát hiện ung thư, bệnh hiểm nghèo) hoặc bảo mật hệ thống. Thà chẩn đoán nhầm (False Positive) còn hơn là bỏ sót bệnh nhân (False Negative).\n",
            "4. **Khi nào Precision quan trọng hơn Recall?**\n",
            "   - Trong lọc thư rác (spam) hoặc hệ thống gợi ý đầu tư tài chính. Việc đánh dấu nhầm một email quan trọng là spam sẽ gây hậu quả rất phiền toái cho người dùng.\n",
            "5. **Vì sao PR-AUC thường hữu ích với dữ liệu imbalance?**\n",
            "   - Vì PR-AUC tập trung trực tiếp vào lớp thiểu số (Positive) và không bị ảnh hưởng bởi số lượng lớn mẫu âm tính (True Negatives) như ROC-AUC.\n",
            "6. **Vì sao StratifiedKFold phù hợp hơn KFold trong bài này?**\n",
            "   - Đảm bảo mỗi fold đều có tỷ lệ mẫu bệnh (lớp thiểu số) đồng đều, tránh tình trạng có fold không có mẫu bệnh nào dẫn tới lỗi mô hình hoặc đánh giá sai.\n",
            "7. **Random Under-Sampling có thể làm mất thông tin như thế nào?**\n",
            "   - Việc loại bỏ ngẫu nhiên các mẫu thuộc lớp đa số có thể vô tình xóa đi các mẫu dữ liệu mang tính ranh giới quan trọng, làm mất đi các phân bố thông tin cốt lõi.\n",
            "8. **SMOTE khác Random Over-Sampling ở điểm nào?**\n",
            "   - Random Over-Sampling chỉ đơn thuần sao chép trùng lặp các mẫu cũ (dễ gây overfitting), trong khi SMOTE tạo ra các mẫu mới hoàn toàn dựa trên phép nội suy giữa các điểm lân cận gần nhất."
        ]
    }
]

# Append the cells
nb_data['cells'].extend(new_cells)

with open(notebook_path, 'w', encoding='utf-8') as f:
    json.dump(nb_data, f, ensure_ascii=False, indent=2)

print("Đã tự động thêm toàn bộ code lời giải và phân tích chi tiết cho 7 bài tập vào cuối file notebook thành công!")

Đã tự động thêm toàn bộ code lời giải và phân tích chi tiết cho 7 bài tập vào cuối file notebook thành công!
